# Sub-Agents

**Prerequisites:** `04_planning.ipynb`, and from `01_foundations`: `01_your_first_llm.ipynb`

`Graph` runs several *tool calls* at once. Every one of them is a single function with known arguments, decided in advance by one planner.

Some work doesn't decompose that way. "Research these two topics and compare them" isn't two tool calls — it's two open-ended jobs, each of which may need several tool calls, a few decisions, and a different number of turns. What that needs isn't a parallel step. It's another agent.

This notebook builds forking: a `Think` that decides to split into sub-agents, each running its own `Runtime` with its own conversation. Then the harder question — what those sub-agents are allowed to see, of the outer conversation and of each other — which turns out to be the difference between delegation, self-consistency, and debate.

**By the end you'll have:**
- Made delegation a decision `Think` reaches through native tool calling, and seen why every field of that schema is paid for on every later call
- Seen a branch that inherited the outer conversation answer *another* branch's question, and fixed it by giving branches nothing but a self-contained goal
- Learned that budgets come in two kinds, and that dividing the wrong one cripples every branch for no benefit
- Declared a fork instead of asking for one, with `Fork` — because a debate's shape is your experiment design, not a judgment for the model
- Used `persistent`, `broadcast` and `start`, and given a `Think` the `advance` tool so it decides who speaks next
- Built `SelectThink`, and `ExpandThink` on top of it: a search whose parents are structural rather than chosen, and which can go back to a line it abandoned
- Left a slow branch running and carried on without it, then collected it when its answer was finally needed

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

from chat import Conversation
from runtime import Runtime, Loop
from observe import Observe
from think import Think, SelectThink, ExpandThink, Fork
from act import Act

In [2]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import BaseModel, field_validator

def get_current_time(timezone):
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_current_time": get_current_time, "convert_temperature": convert_temperature}
SCHEMAS = {"get_current_time": GetCurrentTimeArgs, "convert_temperature": ConvertTemperatureArgs}
tools = [
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {"type": "object",
                       "properties": {"timezone": {"type": "string", "description": "IANA timezone name"}},
                       "required": ["timezone"]}}},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {"type": "object",
                       "properties": {"value": {"type": "number"},
                                      "from_unit": {"type": "string", "enum": ["C", "F"]},
                                      "to_unit": {"type": "string", "enum": ["C", "F"]}},
                       "required": ["value", "from_unit", "to_unit"]}}},
]

## Delegation is a tool call

A `Think` that can fork needs to decide *whether* to fork, *how many* branches, and *what each one should do* — and to do that in the same breath as deciding between an ordinary tool call and a final answer. Making that a separate `ForkThink` class would mean a component that only ever forks, declared in a loop position where forking is the only option.

So `delegate` is just another tool in the list:

```python
def _delegate_tool(conversation):
    return {"type": "function", "function": {
        "name": "delegate",
        "description": "Delegate to independent sub-agents. ..." + budget_note,
        "parameters": {"type": "object", "properties": {
            "persistent": {"type": "boolean",
                           "description": "Optional, default false. True keeps branches alive for another turn later."},
            "broadcast":  {"type": "boolean",
                           "description": "Optional, default false. True lets branches see each other's replies -- needed for a debate."},
            "start":      {"type": "string",
                           "description": "Optional. Id of the single branch to run first, the rest waiting for a turn."},
            "wait_for":   {"type": "array", "items": {"type": "string"},
                           "description": "Optional. Ids you need before carrying on; any others keep running in the background."},
            "branches": {"type": "array", "description": "One entry per sub-agent.",
                         "items": {"type": "object", "properties": {
                "id":         {"type": "string", "description": "Short name, e.g. 'pro'. Needed only when persistent is true."},
                "goal":       {"type": "string", "description": "A complete, self-contained instruction ..."},
                "allow_fork": {"type": "boolean", "description": "Optional, default false. True only if this sub-goal is itself big enough to need splitting again."},
                "explain":    {"type": "boolean", "description": "Optional, default false. True if you will need to see how this branch worked."},
                "temperature": {"type": "number", "description": "Optional. Omit unless several branches share an identical goal."},
                ...
            }}},
        }, "required": ["branches"]},
    }}
```

**Every field carries a description, including the optional ones, and that is not tidiness.** An earlier version described only `goal` and left the rest as bare types. The model filled them in anyway, badly — writing `"broadcast": "false"` as a string into a boolean, and inventing `max_iterations: 1, max_seconds: 10` for a two-step lookup that had no budget to divide. A branch handed `max_iterations: 1` is crippled before it starts.

Measured over 8 attempts on the same request: with bare types, most valid calls invented optional fields; with one short description each, **none did**.

### Every field is paid for on every later call

Verbosity is the opposite failure, and it costs more than it looks. A single ReAct run measured through the API's own `usage`:

```
3 calls | 2394 prompt + 72 completion = 2466 tokens
```

**Ninety-seven percent of that is prompt.** Every turn re-sends the whole conversation *and* every tool schema. A schema is not paid once when it is written; it is paid on every call for the rest of the run.

And the cost is not only tokens. Notebook 4's `replan` tool was first written with its guidance in its own description — 1202 characters, twice the size of the two real tools put together. Offered alongside them, llama-3.3-70b (the model this was measured on) stopped being able to call **any** tool at all: five attempts running, it produced llama's `<function=name{...}</function>` text form instead of a real tool call. Remove `replan` and the identical request succeeded first try.

So prose about *when* and *how* to use a tool goes in the system message, which is plain text and costs the generator nothing:

> Delegating costs a whole extra agent loop per branch. Prefer it for independent sub-goals that each need several steps of their own, for answering one question several times to see if the answers agree, or for arguing opposing positions. For work you could finish in a couple of tool calls, just make the calls.

The schema describes the call's *shape*; the system message describes the *choice*. Tool schemas are a shared budget — `replan` did not break `replan`, it broke `get_current_time`.

### The two kinds of budget

`_delegate_tool` is rebuilt on every call so it can read `conversation.remaining_budget` — the dict notebook 2 computed and nothing used — and say what is actually left. The first version told the model to *"split each of these between branches as whole numbers"*, for all four budgets. That is wrong for half of them.

|  | kind | why |
|---|---|---|
| `tokens`, `tool_calls` | **pooled** | really consumed. Three branches making five calls each is fifteen calls. Must be divided. |
| `seconds`, `iterations` | **per branch** | branches run *concurrently*, so three finish when the slowest finishes, not when their times add up. And each `Runtime` counts its own iterations, so a branch's loop never decrements the caller's. |

Splitting a pool is correct. Splitting a ceiling is nonsense: given 45 seconds and three branches, handing each 15 protects nothing and cripples all three. So the description now says both things separately:

> One shared pot remains, and every branch spends out of it: 3000 tokens, 12 tool calls. Divide it between them in proportion to how much of the work each is doing, and keep some back for answering once they report.

> These are a limit per branch, not a pot to divide, because branches run at the same time: 45 seconds, 6 iterations. Give each branch what its own sub-goal needs, up to that limit, leaving room for the work that follows.

Both end the same way, and that part is your point about the goal: don't hand out everything, because the outer loop still has to read the results and answer afterwards.

### Turning a `delegate` call into a decision

`Think.run()` needs one extra branch — if the tool it picked is `delegate`, the decision is a fork rather than a tool call:

```python
call = message.tool_calls[0]
if call.function.name == "delegate":
    args = json.loads(call.function.arguments)
    return {"role": "decision", "type": "fork",
            "content": "Forked into: " + "; ".join(b["goal"] for b in args["branches"]),
            "persistent": args.get("persistent", False),
            "broadcast": args.get("broadcast", False),
            "start": start, "wait_for": wait_for,
            "branches": args["branches"]}
```

`start` and `wait_for` are validated against the branch ids before the decision leaves here, because `Act` matches on them; a name the model invented is dropped rather than raising from inside execution. Same guarantee `SelectThink` makes about its indices.

**`allow_fork` decides who is offered the tool** — appended per call, not per class. That flag exists because of a real failure: a branch given "get the current time in Tokyo" would sometimes delegate *that* to a sub-sub-agent.

**And it is withheld for one turn after branch results come back.** Handed a pile of results, the model reaches for `delegate` again — and that is the hardest schema here for it to emit, so it burns every retry on malformed syntax and the run dies. The first fix was to declare a *second* `Think` with `allow_fork=False` after the first, which meant every forking loop carried a near-duplicate component. The rule replaces it:

```python
def _can_fork(self, conversation):
    if not self.allow_fork:
        return False
    last = conversation.messages[-1] if conversation.messages else None
    return not (last and last.get("source") == "branches")
```

`Observe` tags the merged branch observation with `source: "branches"`, so the conversation itself says what just happened. One turn, not the whole run — forking again later for genuinely new work is fine.

## Running the branches

`Act` handles a `fork` decision by starting each branch in its own thread, each with its own fresh `Runtime`:

```python
def _fork(self, conversation, decision):
    branches = decision["branches"]
    ids = [b.get("id", f"branch{i}") for i, b in enumerate(branches)]

    with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as pool:
        branch_conversations = list(pool.map(lambda b: self._start_branch(conversation, b), branches))

    results = [{"goal": b["goal"], "id": i, "content": bc.messages[-1]["content"]}
               for b, i, bc in zip(branches, ids, branch_conversations)]
    ...
    return results
```

Same `ThreadPoolExecutor` reasoning as notebook 4, and the same rule as an ordinary tool call: `_fork` doesn't return until every branch is finished. A fork is a blocking call that happens to run several agents inside it.

Where does a branch's `Runtime` come from? `Act` doesn't know what a branch's loop should look like — that's a design decision belonging to whoever declared the outer loop. So `Act` takes a factory:

```python
Act(functions=..., schemas=..., branch_runtime=make_branch_runtime)
```

which receives the branch dict and returns a `Runtime` for it. That's also where the branch's share of the budget and its sampling settings get applied:

In [3]:
def make_branch_runtime(branch):
    return Runtime(
        loop=[
            Think(tools=tools,
                  # both read from the branch dict the delegating Think filled in -- it wrote the
                  # goal, so it is the only thing that knows how big this sub-goal is and whether
                  # the caller will want to see the working
                  allow_fork=branch.get("allow_fork", False),
                  explain=branch.get("explain", False),
                  temperature=branch.get("temperature"), top_p=branch.get("top_p")),
            # `make_branch_runtime` names itself: a branch allowed to fork needs a way to build
            # runtimes for ITS children. Without this, Act would reach for a branch_runtime of
            # None the moment it forked.
            Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=make_branch_runtime),
            Observe(),
        ],
        repeat_from=0,
        max_iterations=branch.get("max_iterations") or 9,
        max_seconds=branch.get("max_seconds"),
        max_tool_calls=branch.get("max_tool_calls"),
        max_tokens=branch.get("max_tokens"),
    )

A branch is an ordinary ReAct loop — notebook 2's, unchanged. `temperature` and `top_p` come straight from chapter 1's notebook 1, threaded through `Think` into `client.chat.completions.create()`; they'll matter later in this notebook.

`max_iterations` falls back to 9 when the delegating `Think` didn't assign one, so a branch is never unbounded.

**Three of these settings come from the branch dict, and that is the point.** `temperature`, `allow_fork` and `explain` are all filled in per branch by the `Think` that delegated — because all three depend on the *goal*, and it wrote the goal.

**`allow_fork` per branch.** A branch given "get the current time in Tokyo" should not delegate that onward; a branch given "research the competitive landscape and summarise it" might reasonably split it. Only something that has read the sub-goal can tell those apart. The default is `False`, so a branch can only fork when the delegator explicitly says so — and depth is bounded by the budget, which each level advertises and divides.

**`explain` per branch, and it is not merely cosmetic.** Notebook 2 has `Think` ask the model to say why it is calling a tool, kept as `decision["reasoning"]`. For a leaf branch that backfires: prompted to narrate, branches answer their goal with an account of what they did rather than the result.

> `[Convert 100 degrees Fahrenheit to Celsius]` *The function call was necessary to get the accurate conversion of temperature from Fahrenheit to Celsius.*

That is the branch's **final answer**, and the number is nowhere in it. So `explain=False` is the right default. But it is only a default — with `branch_detail` (later in this notebook) the caller can read a branch's whole working, and reasons are the most useful part of it. Whether that is wanted depends on the goal, which is why it sits on the branch rather than on the factory.

## What a branch is allowed to see

Now the question that decides whether any of this works.

A sub-agent needs context. The obvious way to give it context is to hand it the conversation so far — it's right there, and it contains everything relevant.

Here is what that does. The outer request has two parts; the branch has been assigned one of them:

In [4]:
OUTER_REQUEST = "What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?"
BRANCH_GOAL = "Get the current time in Tokyo."

outer = Conversation()
outer.messages.append({"role": "user", "content": OUTER_REQUEST})

# the tempting design: seed the branch with everything the outer agent knows
leaky = Conversation()
leaky.messages = list(outer.messages) + [{"role": "user", "content": BRANCH_GOAL}]
make_branch_runtime({}).run(leaky)
print("inherited history ->", leaky.messages[-1]["content"])

inherited history -> The current time in Tokyo is 2026‑08‑18 00:11 JST, and 100 °F is equivalent to 37.8 °C.


In [5]:
# what Act actually does: system messages (if any), plus the branch's own goal, and nothing else
isolated = Conversation()
isolated.messages = [{"role": "user", "content": BRANCH_GOAL}]
make_branch_runtime({}).run(isolated)
print("goal only        ->", isolated.messages[-1]["content"])

goal only        -> The current time in Tokyo is 2026‑08‑18 00:04 JST.


The branch that inherited the conversation answered the temperature question too — a question that was never its job, that another branch was simultaneously working on, and whose answer will now be reported twice to the outer loop as if two things had been confirmed independently.

It isn't confused. It's doing exactly what the context asks: the original two-part request is sitting right there in its history, so it answers both parts.

```python
def _start_branch(self, conversation, branch):
    branch_conversation = Conversation()
    branch_conversation.messages = [m for m in conversation.messages if m["role"] == "system"]
    branch_conversation.messages.append({"role": "user", "content": branch["goal"]})
    self.branch_runtime(branch).run(branch_conversation)
    return branch_conversation
```

A branch gets system messages, if any, and its own goal. Nothing else. Not the outer request, not the other branches' goals, not the reasoning that led to the split.

That pushes the burden onto the goal itself, which is why `_delegate_tool`'s description of that field is so insistent:

> A complete, self-contained instruction for an agent with NO other context — it will not see this conversation or any other branch's goal. Include everything it actually needs (e.g. the specific question or values involved). Never write it as if the reader already knows the broader request, and never reference 'the other task' or another branch.

Writing a good goal *is* the delegation. A `Think` that writes "handle the second part" has not delegated anything, and the isolation makes that failure loud and immediate rather than subtle.

## Decomposition: the one-shot fork

Both flags off. Branches run once, report back, and are discarded.

Before the demo, a correction to something the `delegate` schema didn't say. Its description explained every field in detail and never said **when delegating is worth doing at all** — so the model knew how to fill the form in, but not whether it should. A tool whose cost is never mentioned looks free. Each branch is a whole extra agent loop: its own conversation, its own model calls, its own turns.

So the guidance now says what the choice depends on:

> Delegating costs a whole extra agent loop per branch. Prefer it for independent sub-goals that each need several steps of their own, for answering one question several times to see if the answers agree, or for arguing opposing positions. For work you could finish in a couple of tool calls, just make the calls — unless you were explicitly asked to split the work.

Two things about where that lives. It is **guidance, not structure**: the tempting fix is to hand the outer `Think` no tools so delegating is its only option, but an orchestrator that can do nothing else isn't deciding, it's cornered — and deciding is precisely `Think`'s job. And it sits in the **system message rather than the tool description**, because every word added to a tool description enlarges the JSON schema the model must generate, and a larger schema measurably worsens Groq's tool-call syntax. Prose about *when* to use a tool belongs in prose; the schema should describe the call's shape and nothing else.

### The demo asks to be split, and that is the honest framing

It would be neater to show the model choosing to fork on the merits. It won't, and finding that out is worth more than the neat version.

Given the two-part request below *without* an instruction — and even given a deliberately larger task, two cities each needing a time lookup, a conversion and a judgement — the model reliably declines to delegate and starts calling the tools itself. That is the correct answer. It has both tools; six calls it can make directly will always beat standing up two sub-agents to make them for it. The guidance is working exactly as written.

Which exposes the real precondition: **delegation earns its cost when a branch can do something the caller can't do as easily** — reach different tools, hold a different position, or grind through a long sub-task in its own context rather than filling up yours. Two one-shot tools clear none of those bars. So this demo uses the escape hatch the guidance itself names, and tells it to split:


In [6]:
decompose = Runtime(
    loop=[
        Observe(event=(
            f"{OUTER_REQUEST}\n\n"
            "These are two independent lookups. Delegate them to exactly 2 branches, one per "
            "question, each with a self-contained goal. Then combine what they report into a "
            "single answer."
        )),
        Think(tools=tools, allow_fork=True),
        Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=make_branch_runtime),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=16,
)

result = decompose.run(Conversation())
for m in result.messages:
    print(f"- {m['role']:9} {m['content']!r}\n")

- user      "What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?\n\nThese are two independent lookups. Delegate them to exactly 2 branches, one per question, each with a self-contained goal. Then combine what they report into a single answer."

- decision  "Forked into: Use the get_current_time function to obtain the current date and time in Tokyo, Japan (IANA timezone 'Asia/Tokyo'), and return the result as a string.; Use the convert_temperature function to convert 100 degrees Fahrenheit to Celsius, and return the result as a number."

- tool      "[Use the get_current_time function to obtain the current date and time in Tokyo, Japan (IANA timezone 'Asia/Tokyo'), and return the result as a string.] The current date and time in Tokyo, Japan is **2026‑08‑18\u202f00:04\u202fJST**.\n[Use the convert_temperature function to convert 100 degrees Fahrenheit to Celsius, and return the result as a number.] 37.8"

- decision  'The current time in Tokyo is **2026‑08‑18

The `fork` decision names the goals it invented; `Act` ran them concurrently; `Observe` merged the results into a single `tool` observation, one line per branch:

```python
if isinstance(effect, list):
    return {"role": "tool", "source": "branches",
            "content": "\n".join(render_one(b) for b in effect)}
```

This is the join. Notebook 2's `Observe` handled one effect; a fork produces a list, and the merge is a labelled concatenation — mechanical, with the labels preserving which branch said what so the next `Think` can attribute claims correctly.

Whatever the detail level, it stays **one message**. Splicing a branch's turns into the outer conversation separately would break it twice over: the next `Think` would read them as its own prior turns, and two branches' turns would interleave into nonsense.

**`repeat_from=1` exists so the run cannot end without an answer.** The first version was a flat sequence with no `repeat_from`, on the reasoning that forking once and answering is genuinely a pipeline. But if the final `Think` decides to make one more tool call rather than answer — which it does sometimes, having just been handed a pile of branch results — `Act` executes it and the runtime walks off the end of the list. The run "succeeds" and produces no `assistant` message at all.

**This loop used to have two `Think`s.** A second one, `allow_fork=False`, sat after the merge so the return pass could not try to fork again. It was a near-duplicate component whose only job was to work around a failure — and `repeat_from` had to point past it. The `_can_fork` rule above replaced it: one `Think`, cycled, that simply isn't offered `delegate` on the turn after results land.

## Self-consistency: same goal, different sampling

Every branch so far got a different goal. Nothing requires that.

Give several branches the *same* goal and vary `temperature` and `top_p`, and the fork stops being about dividing work and starts being about disagreement: run the reasoning several times independently, and see whether it lands in the same place. Where it doesn't, you've learned the answer wasn't stable.

Chapter 1's notebook 1 introduced those knobs to control a single answer. Here they're the mechanism for generating *several*, deliberately varied, from one model.

### This one is declared, not asked for

The version of this cell that ran first said, in prose, to the model:

> *"Delegate this exact question to 3 independent branches, each with the identical goal but a different temperature (try 0.2, 0.8, and 1.2). Do not change the wording between branches."*

Read that again. It is a configuration file, written in English, parsed by a model that, when this was written, emitted `delegate` malformed about a quarter of the time — and its most important constraint, *identical wording*, is **pleaded for** rather than guaranteed. Nothing checks it. If the model paraphrases one goal, the experiment is quietly measuring something else.

Three branches at three temperatures is not a judgment. It is the shape of the experiment, and the experiment is yours. So `Fork` emits the same decision `Think` produces for `delegate`, written by the caller instead:

```python
class Fork:
    def run(self, conversation):
        return {"role": "decision", "type": "fork",
                "content": "Forked into: " + "; ".join(b["goal"] for b in self.branches),
                "persistent": self.persistent, "broadcast": self.broadcast,
                "start": self.start, "wait_for": self.wait_for,
                "branches": self.branches}
```

`Act` needed no new code to handle it — same `type`, same fields. The precedent is `Observe(event=...)`: a component handed fixed content that emits it instead of computing one.

A question models often get wrong in a specific, tempting way:

In [7]:
TRICK = ("I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? "
         "Answer with a number and one short sentence of reasoning.")

consistency = Runtime(
    loop=[
        Observe(event=TRICK),
        # identical wording is now true by construction -- one string, three temperatures
        Fork(branches=[{"goal": TRICK, "temperature": t} for t in (0.2, 0.8, 1.2)]),
        Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=make_branch_runtime),
        Observe(),
        Think(tools=tools, allow_fork=False),
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
        Observe(),
    ],
    repeat_from=4,   # the reconciling Think may want a tool call before it can answer
    max_iterations=12,
)

voted = consistency.run(Conversation())
for m in voted.messages:
    print(f"- {m['role']:9} {m['content']!r}\n")

- user      'I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? Answer with a number and one short sentence of reasoning.'

- decision  'Forked into: I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? Answer with a number and one short sentence of reasoning.; I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? Answer with a number and one short sentence of reasoning.; I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? Answer with a number and one short sentence of reasoning.'

- tool      '[I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? Answer with a number and one short sentence of reasoning.] 3 — you still have the three apples you have today; the apple you ate yesterday doesn’t change today’s count.\n[I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? Answer with a number and one short sentence of reaso

Three runs of one question, sampled differently, then a final `Think` reading all three at once and reconciling them.

**There is no voting component.** The merged observation contains every branch's answer, labelled. A `Think` reading three labelled answers and picking the one that recurs is just a `Think` reading its context. Adding a `VoteThink` would have hardcoded one reconciliation policy — majority — when "two agree but the third gives a better reason" is a judgment worth keeping available.

**The model is the same across branches; only sampling differs.** Different models would confound the measurement: a disagreement could mean the reasoning is unstable, or just that two models differ. Holding the model fixed keeps the answer to one question.

**`repeat_from=4` is the same lesson as decomposition's.** The reconciling `Think` holds tools, so it may answer *or* decide on one more call; a flat sequence ending on `Act` leaves the second case with no answer at all. Any loop whose `Think` can call a tool needs somewhere to go back to.

## `persistent`, `broadcast`, and who opens

Both demos so far ran branches once and threw them away. Three settings change that, and the first two are deliberately *not* one `mode` field, because all four combinations are things you'd want.

```python
if broadcast:
    for i, bc in zip(ids, branch_conversations):
        for result in results:
            if result["id"] != i:
                bc.messages.append({"role": "user", "content": f'[{result["id"]}] said: {result["content"]}'})

if persistent:
    conversation.branches.update(zip(ids, branch_conversations))   # merged, never assigned
    conversation.branches_broadcast = broadcast
```

|  | `broadcast=False` | `broadcast=True` |
|---|---|---|
| **`persistent=False`** | Decomposition, self-consistency — the two demos above | Branches see each other once, then end. Rarely what you want, but harmless |
| **`persistent=True`** | Independent specialists you can return to individually, each keeping its own accumulated context | Debate — branches keep going *and* see each other's turns |

**`persistent`** keeps each branch's `Conversation` alive on `conversation.branches`, so it can be resumed later with its history intact rather than restarted from its goal. It is also what makes a set-aside candidate recoverable: pruning only ever filters `pending`, so the branch itself is still there.

That dict is **merged, not assigned**, and the difference is load-bearing. A `Fork` declared inside a loop runs again every round; assigning would delete every branch of every earlier round — taking the whole searched tree with it, which is precisely what has to survive for backtracking.

**`broadcast`** copies each branch's result into every sibling's history. Two details in that loop were both bugs first:

- `results` is captured *before* the broadcast loop runs. Reading `bc.messages[-1]` afterwards returns whatever was just appended to it — a sibling's message — so each branch was reported as having said what the other one said.
- The copy goes in as `{"role": "user", "content": "[other] said: ..."}`, not under the branch's own role. Appended as its own turn, a branch's next call reads the opponent's argument as something it said itself, and promptly agrees with it.

Both are the same underlying mistake: forgetting that a branch's conversation is written from *its* point of view.

### `start`: somebody has to go first

Everything so far ran every branch at once. For a debate that is wrong, and not subtly: if both sides speak simultaneously, **neither is responding to anything**. You get two opening statements and call it an argument.

`start` names the branch that runs now. The others are created, seeded, and left waiting — and because `broadcast` is on, the opener's turn is already in their history when their turn comes:

```python
branch_conversations = [self._seed_branch(conversation, b) for b in branches]
first = ids.index(start)
self.branch_runtime(branches[first]).run(branch_conversations[first])
ran = [(ids[first], branches[first], branch_conversations[first])]
```

Seeding and running had to be separated for this — every branch is fully formed, only one has run.

It is `start` and **not** a running order, and that distinction is the whole design. An order would decide every turn in advance. Who speaks *second* depends on what the first side actually said, which is a judgment, and judgments belong to a `Think`.

In [5]:
SIDES = [
    {"id": "python", "goal": "Should a small startup write its backend in Python or in Go? "
                             "Argue the case for Python."},
    {"id": "go",     "goal": "Should a small startup write its backend in Python or in Go? "
                             "Argue the case for Go."},
]

debate = Runtime(
    loop=[
        Observe(event="Should a small startup write its backend in Python or in Go?"),
        # declared, not asked for: two sides, both kept alive, both hearing each other,
        # and python opens
        Fork(branches=SIDES, persistent=True, broadcast=True, start="python"),
        Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=make_branch_runtime),
        Observe(),
    ],
    max_iterations=8,
)

arena = debate.run(Conversation())
for branch_id, bc in arena.branches.items():
    spoke = [m for m in bc.messages if m["role"] == "assistant"]
    print(f"[{branch_id}] {'opened: ' + spoke[-1]['content'][:110] + '...' if spoke else 'has not spoken yet'}")
print()
print("branches kept alive:", list(arena.branches), "| broadcasting:", arena.branches_broadcast)

[python] opened: **Why a Small Startup Should Choose Python for Its Backend**

---

### 1. Speed of Development (Time‑to‑Market...
[go] has not spoken yet

branches kept alive: ['python', 'go'] | broadcasting: True


Both branches exist and both are addressable on `arena.branches` — but only `python` has spoken. `go` is seeded, unrun, and already holds python's opening as a `user` turn in its own history. When it does speak, it will be answering.

### Resuming a branch, and choosing which

```python
def _advance_branch(self, conversation, decision):
    branch_conversation = conversation.branches[decision["branch"]]
    if decision.get("message"):
        branch_conversation.messages.append({"role": "user", "content": decision["message"]})
    self._run_branch({"goal": None}, branch_conversation, getattr(conversation, "abandoned", None))
    ...
```

A fresh `Runtime` over an existing conversation — the branch's accumulated history is what makes it a continuation rather than a restart. A turn is a full run of the branch's own loop, not one message: responding to an argument may take several tool calls.

**Deciding *which* branch goes next is a `Think`'s job**, reached the same way a fork is — through a synthetic tool, offered only once persistent branches exist:

```python
if self.allow_fork:
    all_tools.append(_delegate_tool(conversation))
if self.allow_advance and getattr(conversation, "branches", None):
    all_tools.append(_advance_tool(conversation))
```

**Those two flags used to be one, and merging them made a whole component unbuildable.** A debate moderator should be able to advance a branch and *never* start a new one — a fresh branch begins with none of what the existing one established. But while `advance` lived inside `if self.allow_fork:`, turning delegation off took turn-taking with it. `allow_advance` now defaults to `allow_fork`, so nothing that already worked changed, and `Think(allow_fork=False, allow_advance=True)` finally means something.

**The tool itself is deliberately small** — a name and an optional message, with the branch ids listed. Its first version explained when to advance and why, inside its own description, and that is exactly the mistake `replan` made: 715 characters that every other tool in the list pays for on every call. When to prefer advancing lives in the system message:

> Branches already exist: python, go. To continue an exchange, press a point, or revisit an answer you set aside, use `advance` on one of them rather than delegating again — a new branch would start with none of what that one has already established.

That sentence exists because, offered both tools, the model reached for `delegate` — spinning up a *new* branch to write a rebuttal and discarding everything the existing one had said.

In [6]:
# the debate is under way; go has not spoken yet. Let a Think pick who continues --
# allow_fork=False so it cannot start a new side, allow_advance=True so it can pick one.
arena.messages.append({"role": "user", "content":
    "Continue the debate: give whichever side has the most to answer for another turn."})

turn = Think(tools=None, allow_fork=False, allow_advance=True).run(arena)
arena.messages.append(turn)
print("decision:", turn["type"], "->", turn.get("branch"))
print("told to :", turn.get("message"))
print()

Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=make_branch_runtime).run(arena)
print(Observe().run(arena)["content"][:600])

decision: advance_branch -> go
told to : None



[go] **Why a Small Startup Should Choose Go for Its Backend**

---

### 1. Speed‑to‑Market Through Rapid Development

| Aspect | Go | Python |
|--------|----|--------|
| **Compilation‑time feedback** | Instant compile‑run cycle; the compiler catches many bugs before you even start the server. | Errors surface at runtime, which can lead to nasty production bugs that waste time fixing. |
| **Static typing** | Enforces contracts early, reducing the need for extensive unit‑test scaffolding on simple data‑flow code. | Dynamic typing means you often have to write extra tests just to catch type‑relat


The `Think` read the exchange, picked a side by id, and wrote what to put to it — none of which was hardcoded. The branch continued from its own history, so it answers in context rather than from a restated goal.

**It has no tools of its own**, and that is the same reasoning as ReWOO's solver: a moderator picks who speaks and judges when the exchange has settled. Looking things up is the branches' work. Handing it the tools as well is pure downside — more schema on every call, and more chance of a malformed one.

**It could only name what it had seen.** `Observe` used to label branch results by *goal*, so the transcript said `[argue for the case for Python]` while the branch was called `python`. Asked to advance one, the model naturally used the label it had read, and the id didn't exist. Persistent branches are now labelled by id — the id is the handle, and a component asked to name one can only use what the conversation showed it.

Judgement is still judgement, and it has limits: a run of this cell asked the *Go* side how **Python** answers concerns about performance, so Go spent its turn explaining Python's position. Picking who speaks is mechanised; picking what to ask them is not.

**And an exchange has no natural end.** Cycled with `repeat_from`, this `Think` will advance a branch every single turn until the budget stops it — asked to wrap up, one run told a branch to *"summarize the main points"*, delegating even the conclusion. Notebook 6's Debate therefore bounds the exchange with `Loop(count=turns)` and follows it with a `Think` that has **no** `advance` tool, so summing up is the only move left. Telling a model when to stop is a suggestion; taking the tool away is a fact.

## Choosing between branches: `SelectThink`

`Observe` merges every branch result into the conversation. When branches divided up work, keeping all of them is right — each holds a different piece.

When branches are *competing* — several attempts at the same thing — keeping all of them is waste. Every future turn re-reads candidates already known to be worse.

```python
class Selection(BaseModel):
    keep_indices: list[int]
    scores: list[float] = []          # one per candidate, in order
    reasoning: str

class SelectThink(Think):
    def run(self, conversation):
        candidates = getattr(conversation, "pending", None)
        candidates = candidates if isinstance(candidates, list) else []
        listing = "\n".join(f"{i}. [{c.get('goal', '(no goal)')}] {c['content']}"
                            for i, c in enumerate(candidates))
        result = extract(
            f"Conversation so far:\n{render(conversation)}\n\n"
            f"Candidate results from {len(candidates)} independent branches:\n{listing}\n\n"
            f"Score each against the original goal out of 10, giving one score per candidate in "
            f"the order listed above, and pick the best {self.keep} to continue exploring, by "
            "their index in the list above. Discard the rest.\n"
            f"Return JSON matching this schema:\n{Selection.model_json_schema()}",
            Selection,
        )
        valid = [i for i in dict.fromkeys(result.keep_indices) if 0 <= i < len(candidates)]
        scores = {}
        if len(result.scores) == len(candidates):
            scores = {c["id"]: s for c, s in zip(candidates, result.scores) if c.get("id")}
        return {"role": "decision", "type": "prune", "content": result.reasoning,
                "keep_indices": valid[: self.keep] or list(range(min(self.keep, len(candidates)))),
                "scores": scores}
```

**The schema is embedded, and that was a bug first.** The prompt originally ended "Return JSON matching the given schema" without ever including the schema. Asked to return JSON with no field names given, the model reasonably invented its own — a list of scored candidates under a key like `taglines` — and `Selection` rejected every one. It sometimes recovered, because `extract()` feeds each validation error back, but "sometimes" is not a design.

**The indices are guaranteed here, not trusted.** `Act` indexes straight into `pending`, and the model returns duplicates, more than it was asked for, and — as this notebook found on its first run, with an `IndexError` — indices that don't exist. So `SelectThink` deduplicates, drops anything out of range, truncates to `keep`, and falls back to the first `keep` if nothing usable survives. Validating in `Act` instead would mean `Act` reasoning about a decision it was handed.

**The scores are kept against branch ids, and that is what makes a search able to change its mind.** Indices are per-round and meaningless once the round is over; an id stays valid as long as the branch does. A discarded branch keeps its score — the loser of round one may still outscore everything in round three. Pruning only ever filtered `pending`, so it was always still *there*; what was missing was any record of how good it had been.

`scores` is optional in the model. Choosing what survives is the critical path; the scores only enrich a later backtrack. Required, a model that omitted them would fail the whole selection and burn `extract()`'s retries over an extra.

`Act` applies the prune, and the application is an index filter:

```python
if decision["type"] == "prune":
    if decision.get("scores"):
        conversation.branch_scores.update(decision["scores"])
    pending = conversation.pending
    if isinstance(pending, list):
        conversation.pending = [pending[i] for i in decision["keep_indices"]]
        conversation.frontier = [b["id"] for b in conversation.pending if b.get("id")]
```

`frontier` is what the next round grows from, written here rather than read from `pending` — because `Observe` consumes `pending`, and by the time the loop comes round again it is gone.

## Searching: `ExpandThink`

`keep=1` narrows to a single winner. `keep=2` or more keeps a *frontier*, and then the loop can expand what survived — which is beam search, and by extension Tree of Thoughts.

The first version of this section did it with the machinery already built: `Think(allow_fork=True)`, told in prose to "delegate again to refine the survivors". That works, and it hides a mistake.

### Who decides what a new thought grows from

Asked to expand a frontier, that `Think` writes several new goals and — if you want a tree at all — has to say which survivor each one came from. So the goal and its parent are two separate fields it fills in, and **nothing can check that they agree.** It can write a thought inspired by `b2` and label it `b1`. The child then opens with the wrong branch's working, and no test can catch it, because only the model ever knew where the thought came from.

That is a choice masquerading as a fact. `ExpandThink` turns it back into a fact by asking **once per surviving branch**, showing that branch's working and nothing else:

```python
def _from_parent(self, conversation, parent_id):
    parent = conversation.branches[parent_id]
    answer = next((m["content"] for m in reversed(parent.messages)
                   if m.get("role") == "assistant"), "")
    working = transcript({"label": parent_id, "content": answer, "conversation": parent})
    return [{"parents": [parent_id], "thought": t} for t in self._ask(
        f"The overall task:\n{render(conversation)}\n\n"
        f"One line of work so far:\n{working}\n\n"
        f"Give {self.thoughts} distinct ways to continue THIS line -- each a complete, "
        "self-contained instruction for an agent that will see the working above."
    )]
```

The parent is recorded, not chosen. A wrong parent stops being unlikely and becomes impossible. It costs one model call per surviving branch instead of one for the whole round — they run in parallel, so it is not slower — and it is what the Tree of Thoughts paper does: its generator is handed one state.

### What a child starts from

Not the parent's raw messages, which would drag every intermediate turn along, but the compacted block `transcript()` renders — goal, calls, why, results, answer. That block *is* the inherited state:

```python
branch_conversation.messages.append({"role": "user", "content": (
    "An earlier line of work you are continuing from:\n" + transcript(...))})
branch_conversation.messages.append({"role": "user", "content": assignment["thought"]})
```

Which is why a search branch runs with `explain=True`. Strip the reasons out and the child inherits calls and results with the most useful part missing.

`Act` allocates the ids — `b0`, `b1`, … numbered on from however many branches already exist — so rounds never collide and every earlier branch stays addressable. That, plus the scores `SelectThink` now keeps, is the whole of backtracking: a line dropped in round one at 6/10 is still there, still scored, still growable in round three.

### The two numbers are yours

`thoughts` is the branching factor and `keep` is the beam width, exactly like Self-Refine's `refine_rounds`. Change them and it is still the same search.

`max_parents` is the one that changes what kind of search it is. At 1, no merge is ever offered, every thought has exactly one parent, and the shape can only split — a tree. Above 1, each round asks one extra question, *"is any group of these worth combining into a single stronger idea?"*, and a yes produces a node with several parents. That is the aggregation edge, and it is the only thing Graph of Thoughts has that Tree of Thoughts doesn't.

Merging is the one thing that cannot be structural: nothing about the loop can work out which two ideas are worth joining. Which is how the paper divides it too — `Generate` takes one thought, only `Aggregate` takes several.

In [5]:
IDEA = "Come up with a memorable tagline for a new brand of running shoes."

def searching_branch_runtime(branch):
    # explain=True is not optional in a search: every child inherits its parent's compacted
    # working as the state it grows from, and with explain off that block carries calls and
    # results but none of the reasons.
    return Runtime(
        loop=[
            Think(tools=tools, allow_fork=False, explain=True),
            Act(functions=FUNCTIONS, schemas=SCHEMAS),
            Observe(),
        ],
        repeat_from=0,
        max_iterations=6,
    )

beam = Runtime(
    loop=[
        Observe(event=IDEA),
        ExpandThink(thoughts=2, max_parents=1),        # <- 1 makes it a tree
        Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=searching_branch_runtime),
        SelectThink(keep=2),
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
        Observe(),
        # a search needs a way out: ExpandThink always expands and SelectThink always prunes,
        # so without this the loop runs until the budget stops it
        Think(tools=tools, allow_fork=False),
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
    ],
    repeat_from=1,
    max_iterations=20,
)

searched = beam.run(Conversation())
print("branches:", list(getattr(searched, "branches", {}) or {}))
print("scores:  ", getattr(searched, "branch_scores", {}))
print("parents: ", getattr(searched, "branch_parents", {}))
print()
for m in searched.messages:
    print(f"- {m['role']:9} {str(m['content'])[:150]!r}")

branches: ['b0', 'b1']
scores:   {'b0': 9.0, 'b1': 8.0}
parents:  {'b0': [], 'b1': []}

- user      'Come up with a memorable tagline for a new brand of running shoes.'
- decision  'Create a concise, high‑energy tagline for the new running‑shoe brand that highlights speed and freedom, using a strong action verb and the brand name '
- decision  'Both taglines satisfy the request for a memorable phrase, but they emphasize different strengths. Candidate\u202f0 uses a strong action verb, includes a br'
- tool      '[b1] **“Run Green, Share the Scene.”**'
- decision  '**“Sprint Light, Feel the Flight.”**'
- assistant '**“Sprint Light, Feel the Flight.”**'


`ExpandThink` wrote two opening thoughts, both ran, `SelectThink` scored them — `{'b0': 9.0, 'b1': 8.0}` — and the finishing `Think` decided that was enough and answered. **One round.**

That is worth being plain about, because the machinery is built for more. `parents` shows `{'b0': [], 'b1': []}`: two roots and no children, so nothing above was inherited by anything, and no line was abandoned and returned to. A two-thought tagline search is simply not hard enough to need a second round, and forcing one by telling the model to keep going would be putting the topology back in the prompt — the thing this notebook has spent its length taking out.

What the run does establish is that every piece works: thoughts generated per line, branches allocated and run concurrently, candidates scored against their ids, the survivor merged, and a `Think` deciding the search was done. Notebook 6 runs the same declaration as Tree of Thoughts and Graph of Thoughts, differing by one number.

**Two things about the scores took a live run each to get right.**

The first version asked for them as a bare parallel array — *"one score per candidate, in the order listed"*. The model returned `[869.0]`: one number for three candidates, and not out of ten either. The length check discarded it, so `branch_scores` came back empty and backtracking had nothing to steer by. Scores are now **labelled** — `{"index": 0, "score": 8.0}` — so a short or reordered list is still usable and a wrong index is detectable. Positional conventions are ones the model has to hold in its head, and it doesn't; the same lesson as labelling branches by id rather than by goal.

The second showed up in the same run: `keep_indices` came back as `[20]` against three candidates, pure garbage, while the scores were sound. The old fallback was "keep the first N", which would have thrown away the best candidate while holding a 9-out-of-10 score for it. Now an unusable selection falls back to **ranking by score**, and only with neither does order become all there is.

**The final `Think` is what ends the search, and it had to be added.** Run without it, this loop expanded and pruned until `max_iterations` fired, and every single run finished on `Stopping: reached the maximum` — never on an answer. `ExpandThink` always expands and `SelectThink` always prunes; neither can decide the search is done. Knowing when to stop searching is a judgment, and the search cannot make it.

## Bringing a branch's working back, not just its answer

Every fork so far reported one line per branch — whatever the branch said last. For splitting up work that is the right amount: the caller asked for an answer and got one.

It is not enough when the caller has to *judge* the work. Did the branch actually check, or guess? Which tool did it reach for? Did it hit an error and route around it?

`Act` keeps the branch's whole `Conversation` on the result — in `conversation.pending`, never in `messages`, so a long branch history costs nothing unless something asks for it. Asking is `Observe`'s job:

```python
def _render(self, branch):
    if self.branch_detail is None:
        detailed = branch.get("explain", False)      # what the delegating Think asked for
    else:
        detailed = self.branch_detail == "transcript"
    return self._transcript(branch) if detailed else self._answer(branch)
```

**The default is `None`, and that is not laziness — it removes a decision that was being made twice.** `explain` is set per branch by the `Think` that delegated it, and asking for a branch's reasoning *is* asking to read its working. Requiring the loop declaration to say `branch_detail="transcript"` as well let the two disagree, silently, in both directions:

- `explain=True` with a default `Observe` — every branch generated a rationale for every call, and the caller was shown only the last line. Tokens spent inside the branch, then thrown away.
- `branch_detail="transcript"` over `explain=False` — a transcript of calls and results with the *reasons missing*, which is the half worth reading.

It also fixes a granularity mismatch. One `Observe` handles a whole fork, but `explain` is **per branch** — so a fork of three where only one needs auditing could not be expressed at all. Now each branch renders by its own setting, and `branch_detail` is an override for when the caller genuinely wants to force one or the other.

Whichever you get, the result is **one message**.

In [6]:
audited = Runtime(
    loop=[
        Observe(event=(
            f"{OUTER_REQUEST}\n\n"
            "Delegate these to exactly 2 branches, one per question, each with a self-contained "
            "goal. Set explain=true on the branch whose working you want to be able to check."
        )),
        Think(tools=tools, allow_fork=True),
        Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=make_branch_runtime),
        Observe(),          # branch_detail=None: each branch rendered by its own `explain`
    ],
    max_iterations=10,
)

seen = audited.run(Conversation())
print(seen.messages[-1]["content"])

[Get the current date and time in Tokyo (IANA timezone 'Asia/Tokyo') using the get_current_time tool and output the result.] The current date and time in Tokyo (Asia/Tokyo) is **2026‑08‑18 00:12 JST**.
[Convert 100 degrees Fahrenheit to Celsius using the convert_temperature tool and output the result.]
  goal: Convert 100 degrees Fahrenheit to Celsius using the convert_temperature tool and output the result.
  called convert_temperature({"from_unit":"F","to_unit":"C","value":100})
  got 100 °F is equal to 37.8 °C.
  answer: 100 °F is equal to 37.8 °C.


One `tool` message. A branch the `Think` marked `explain=true` comes back as its whole working — the goal it was given, the call it chose and why, what came back, and what it concluded — while a branch it didn't comes back as a single line. Two renderings in one message, because the `Think` made two different decisions.

The cost of a transcript is context: a branch that made ten calls puts ten lines into the outer conversation, on every turn thereafter. That is the trade, and it is now made once, per branch, by the thing that wrote the goal.

## Not waiting for everything

Every fork so far blocked until the last branch finished. `_fork` runs them in a `with ThreadPoolExecutor(...)` block, and leaving that block is what waits — the same rule as an ordinary tool call: it doesn't return until complete.

That is wasteful when branches have very different lengths. Two quick lookups and one long piece of research: the outer loop could be working with the lookups while the research continues.

So `delegate` and `Fork` take **`wait_for`** — the ids you need before carrying on. The rest keep running:

```python
waited = [i for i in ids if i in wait_for] or ids
concurrent.futures.wait([futures[i] for i in waited])
for i, b in zip(ids, branches):
    if i not in waited:
        conversation.running[i] = futures[i]
        self._reserve(conversation, i, b)
```

This is the one place the rule "`Act` does not return until everything it started has finished" is deliberately broken, which is why the pool has to move onto the conversation and outlive the call.

**A background branch never writes itself into the conversation.** It appears only when a `Think` decides to `collect` it. That matters more than it sounds: the transcript is currently a function of decisions alone — same decisions, same messages, every time — and that is what makes a bad run debuggable. If a finishing branch could splice itself in whenever it happened to land, two identical runs would differ by timing.

**Only the pooled budgets are reserved.** A branch promised 3000 tokens hasn't spent them yet, so the next fork must not be offered them twice; seconds and iterations are ceilings, and a branch still running has taken wall clock from nobody. `_remaining_budget` subtracts what is reserved, so the figure advertised is *unpromised* budget rather than merely unspent.

**And the run cannot walk away from work it is paying for.** Every exit from `Runtime.run` drains: a shared stop flag is set, reservations are released, the pool is shut down without waiting. Threads cannot be killed, but a `Runtime` checks `cancel` every iteration, so a branch stops at its next clean boundary — one that would have run 200 turns stopped after 1. The flag is inherited by every child and grandchild, so a branch blocked waiting on its own sub-branches unwinds from the leaves up.

In [4]:
import time

# A slow tool, exactly as notebook 4 did to make overlapping steps measurable: real work waits on
# the network, and without something that waits there is nothing here to see.
def research(topic):
    time.sleep(8)                     # stand-in for a slow API
    return f"Findings on {topic}: three sources agree it is well established."

class ResearchArgs(BaseModel):
    topic: str

SLOW_FUNCTIONS = {**FUNCTIONS, "research": research}
SLOW_SCHEMAS = {**SCHEMAS, "research": ResearchArgs}
slow_tools = tools + [{"type": "function", "function": {
    "name": "research",
    "description": "Research a topic in depth. Slow -- takes several seconds.",
    "parameters": {"type": "object", "properties": {"topic": {"type": "string"}},
                   "required": ["topic"]}}}]

def staged_branch_runtime(branch):
    return Runtime(
        loop=[Think(tools=slow_tools, allow_fork=False),
              Act(functions=SLOW_FUNCTIONS, schemas=SLOW_SCHEMAS),
              Observe()],
        repeat_from=0, max_iterations=9, max_tokens=branch.get("max_tokens"))

# Run the components by hand rather than through a Runtime -- as the `advance` demo above does --
# because a Runtime DRAINS on the way out: every exit from run() sets the cancel flag, clears
# `running` and releases reservations, precisely so a finished run never leaves work it is paying
# for. Which means background work is only ever visible from INSIDE a run.
staged = Conversation()
staged.messages.append({"role": "user", "content": "I need two lookups now, research later."})
staged.remaining_budget = {"tokens": 40000, "tool_calls": None, "seconds": None, "iterations": None}
staged.messages.append(
    Fork(branches=[
            {"id": "time", "goal": "What is the current time in Tokyo? Use the tool."},
            {"id": "temp", "goal": "What is 100 degrees Fahrenheit in Celsius? Use the tool."},
            {"id": "deep", "goal": "Use the research tool to research the history of running "
                                   "shoes, then summarise what it returns.",
             "max_tokens": 3000},
         ],
         persistent=True,
         wait_for=["time", "temp"]).run(staged))       # carry on without `deep`

act = Act(functions=SLOW_FUNCTIONS, schemas=SLOW_SCHEMAS, branch_runtime=staged_branch_runtime)
started = time.perf_counter()
act.run(staged)
print(f"Act returned after {time.perf_counter() - started:.1f}s")
print("still running :", list(staged.running))
print("reserved      :", staged.reserved_budget)
print("came back now :", [r["id"] for r in staged.pending])
print()

# ...and later, when its answer is finally needed
staged.messages.append({"role": "decision", "type": "collect",
                        "content": "Waiting for: deep", "branches": ["deep"]})
started = time.perf_counter()
act.run(staged)
print(f"collect waited {time.perf_counter() - started:.1f}s")
print("still running :", list(staged.running))
print("reserved      :", staged.reserved_budget)
print("collected     :", staged.pending[0]["content"][:150])

Act returned after 2.6s
still running : ['deep']
reserved      : {'tokens': 3000, 'tool_calls': 0}
came back now : ['time', 'temp']



collect waited 8.1s
still running : []
reserved      : {'tokens': 0, 'tool_calls': 0}
collected     : Three sources agree that the history of running shoes is well established.


`Act` came back in about three seconds with the two lookups, while `deep` was still inside its eight-second call — and `deep`'s 3000 tokens stayed **reserved**, unavailable to any later fork for exactly as long as it might still spend them. Then `collect` waited out the remainder, returned its answer, and released the reservation back to zero.

**Why this cell drives the components by hand.** A `Runtime` *drains* on the way out: every exit from `run()` sets a shared cancel flag, clears `running`, and releases reservations, so a finished run never walks away from work it is still paying for. Which means a completed run can never show you background work — the first version of this cell inspected `conversation.running` after `staged.run(...)` returned and printed an empty list every time. Background work is only visible from inside a run.

Collecting is a decision like any other. `Think` is offered a `collect` tool while anything is still in flight, and its description lists **what each running branch is working on**, not just its id:

> Still running: deep (Use the research tool to research the history of running shoes...). Use this the moment one of them holds something you need — carrying on without it, or delegating the same work again, wastes the budget it is already spending.

The goal is there because *"do I need `deep` now?"* is unanswerable from an id alone. Deciding to wait is a judgment about what the branch is doing, so the thing making it has to know.

## What you built

✓ Made forking a decision reached through the normal tool-calling path — `delegate` as one more tool — with `allow_fork` per instance and per branch, so the `Think` that wrote a sub-goal decides whether that sub-goal may split again

✓ Measured what a schema costs: 97% of a run's tokens are prompt, re-sent on every call — and watched a 1202-character `replan` description stop the model calling *any* tool at all, five attempts running. Prose about a tool belongs in the system message; the schema describes the call's shape

✓ Learned that budgets come in two kinds — tokens and tool calls are a shared pot to divide, seconds and iterations are a ceiling each branch faces alone, because branches run at the same time — and that the first version told the model to divide all four

✓ Watched a branch seeded with the outer conversation answer a sibling's question, and fixed it by giving branches only a system message plus a self-contained goal

✓ Replaced a paragraph asking the model to "delegate to exactly 2 persistent branches that broadcast" with `Fork` — because three branches at three temperatures is your experiment design, not a judgment, and identical wording should be guaranteed by a comprehension rather than pleaded for in prose

✓ Ran a real debate: `persistent` and `broadcast` as independent flags, `start` naming who opens so the second speaker is answering rather than opening too, and `advance` letting a `Think` pick who continues — a decision `allow_advance` had to be split off `allow_fork` to make expressible at all

✓ Built `SelectThink`, scoring branch results in `pending` before `Observe` can merge them, and keeping those scores against branch ids so a line abandoned in round one is still reachable in round three

✓ Built `ExpandThink`, where a new thought's parent is settled by which line the model was shown rather than named alongside it — turning a wrong parent from unlikely into impossible — with `max_parents` as the single number separating a tree from a graph

✓ Left a slow branch running with `wait_for`, carried on without it, kept its budget reserved until it was collected, and made sure a finishing run cancels what it is no longer going to read

**Next:** `06_agent_patterns.ipynb` — the whole catalogue, every pattern written as a declared loop over the components these five notebooks built, and every one of them run against the real model.